# Sector-Specific Applicant Name Harmonization (Stage 2 Rectification)

Modular post-processing pipeline that takes the generic Stage 1 harmonization output (`harmonization_lookup-2.csv`) and the raw PATSTAT applicant table (`qc_applicants_PROD.csv`), partitions organizations by entity category (`company`, `university`, `gov / non-profit`), and decomposes oversized clusters using squared-IDF weighting and distinguishing-core guards.

In [1]:
%pip install rapidfuzz sentence-transformers networkx


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import networkx as nx
import networkx.algorithms.community as nx_comm
from rapidfuzz.fuzz import ratio as rf_ratio, token_sort_ratio as rf_token_sort
from sentence_transformers import SentenceTransformer

# --- File Paths ---
RAW_APPLICANTS_CSV = "qc_applicants_PROD.csv"
STAGE1_LOOKUP_CSV = "harmonization_lookup_stage1.csv"
OUTPUT_LOOKUP_CSV = "harmonization_lookup.csv"
OUTPUT_FULL_CSV = "harmonization_output.csv"

# --- Model & Rectification Hyperparameters ---
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
MIN_CLUSTER_SIZE_TO_SPLIT = 2     # only decompose Stage 1 blocks with >= 6 distinct names
SCORE_THRESHOLD = 0.56            # aligned with Stage 1 validated Otsu threshold (~0.559) better double check this later
CORE_JAC_MIN = 0.35               # minimum squared-IDF overlap on distinguishing core tokens
CORE_EDIT_MIN = 0.80             # minimum core character similarity if core overlap < 0.35
LOUVAIN_RESOLUTION = 1.35
LOUVAIN_SEED = 42

## 1. Entity-Type Classification & Filtering
Classify applicants using PATSTAT `psn_sector` first, falling back to multilingual regex patterns when `psn_sector` is missing or `UNKNOWN`[cite: 1]. Drop individuals and `-NOT AVAILABLE-` records so only organizations enter harmonization[cite: 1].

In [3]:
LEGAL = [
    "INC", "LLC", "LLLP", "LL", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "COPMANY",
    "GMBH", "AG", "SE", "SA", "SAS", "SARL", "SRL", "SPA", "SL", "BV", "NV", "NA", "OY", "AB", "AS",
    "ASA", "PTY", "PLC", "KK", "LLP", "LP", "PTE", "KG", "OOO", "SP Z OO", "SPOLKA", "JOINT.STOCK",
    "ANONIM SIRKETI", "OBSHCHESTVO"
]
LEGAL_CJK = ["有限公司", "株式会社", "股份", "公司", "会社", "주식회사", "インコーポレイテッド", "アクチボラグ"]
UNIV = [
    "UNIV", "COLLEGE", "POLYTECHN", "POLITECNIC", "ECOLE", "HOCHSCHULE", "SCHOOL",
    "INSTITUTE OF TECHNOLOGY", "GIJUKU", "TRUSTEES", "EDUCATIONAL FUND",
    "大学", "大學", "学院", "カレッジ", "義塾", r"\bETH\b", r"\bEPFL\b", r"\bUESTC\b"
]
GOV = [
    "INSTIT", "INSTYTUT", "INSTUTUC", "ИНСТИТУТ", "ACADEM", "COMMISSARIAT", "CNRS", "LABORATORY",
    "CENTER", "CENTRE", "CENTRUM", "ZENTRUM", "FOUNDATION", "FONDAZIONE", "FUNDACIO", "STIFTUNG",
    "FRAUNHOFER", "FROUNHAV", "PLANCK", "GOVERNMENT", "MINISTRY", "MINISTERIUM", "BUNDES",
    "FEDERATSIYA", "AGENTSTVO", "UCHREZHDENIE", "SECRETARY", "AGENCY", "COUNCIL", "CONSEJO",
    "CONSIGLIO", r"\bNATIONAL\b", "IDRYMA", "ASSOCIATION", "HOSPITAL", "ARMY", "AIR FORCE", "NAVY",
    "RIKEN", r"\bTNO\b", "NASA", "研究", "中心", "委员会", "科学院", "センター"
]
FIRM = [
    "TECHNOLOG", "QUANT", "COMPUT", "SYSTEM", "SOLUTION", r"\bLABS?\b", "SOFTWARE", "NETWORK",
    "ELECTRONIC", "SEMICONDUCT", "PHOTONIC", "DEVICES", "INDUSTR", "HOLDING", "GROUP", "CAPITAL",
    "VENTURES", "ENTERPRISE", "INTERNATIONAL", "SCIENCE", "DIGITAL", "DATA", "INTELLIGEN", "ENERGY",
    "PHARMA", "SECUR", "MACHINES", "INSTRUMENTS", "ENGINEERING", "INNOVATION", "BLOCKCHAIN",
    "BANK", "INSURANCE", "PORTFOLIO", r"\bAI\b", "&", r"\+", "科技", "技术", "量子"
]

LEGAL_RE = re.compile(r"\b(" + "|".join(LEGAL) + r")\b|" + "|".join(LEGAL_CJK))
UNIV_RE, GOV_RE, FIRM_RE = (re.compile("|".join(words)) for words in (UNIV, GOV, FIRM))
WORD = r"[^\W\d_]+(?:['\-][^\W\d_]+)*"
PERSON_COMMA = re.compile(rf"^{WORD}( {WORD}){{0,3}} ?, ?{WORD}")
PERSON_WORDS = re.compile(rf"^{WORD}( {WORD}){{1,3}}$")

def normalize_for_type(name):
    s = unicodedata.normalize("NFKC", str(name))
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)).upper()
    s = " ".join(s.replace("?", "").replace(".", "").split())
    return s.replace("UNIVERSITY CORPORATION", "UNIVERSITY")

def classify_entity(psn_name, psn_sector=None):
    s = str(psn_sector).upper() if pd.notna(psn_sector) else ""
    if "UNIVERSITY" in s:             return "university"
    if "GOV" in s or "HOSPITAL" in s: return "gov / non-profit"
    if "COMPANY" in s:                return "company"
    if "INDIVIDUAL" in s:             return "individual"

    n = normalize_for_type(psn_name)
    if LEGAL_RE.search(n): return "company"
    if UNIV_RE.search(n):  return "university"
    if GOV_RE.search(n):   return "gov / non-profit"
    if FIRM_RE.search(n):  return "company"
    if PERSON_COMMA.match(n) or PERSON_WORDS.match(n) or "·" in n:
        return "individual"
    if len(n.split()) == 1: return "company"
    return "uncertain"

# Load and merge Stage 1 lookup with raw PATSTAT sector metadata
raw_df = pd.read_csv(RAW_APPLICANTS_CSV)
raw_meta = raw_df[["psn_name", "psn_sector"]].drop_duplicates("psn_name")
stage1_df = pd.read_csv(STAGE1_LOOKUP_CSV)

df_work = stage1_df.merge(raw_meta, on="psn_name", how="left")
df_work["entity_type"] = [classify_entity(n, sec) for n, sec in zip(df_work["psn_name"], df_work["psn_sector"])]

# Apply targeted manual overrides for edge cases
df_work.loc[df_work["psn_name"].isin([
    "DHITECH - DISTRETTO TECNOLOGICO H IGH TECH S.C.A R.L.",
    "PEARL COHEN ZEDEK LATZER BARATZ"
]), "entity_type"] = "company"
df_work.loc[df_work["psn_name"].str.contains("EHNSEHRM", na=False), "entity_type"] = "gov / non-profit"

df_work = df_work[(df_work["entity_type"] != "individual") & (df_work["psn_name"] != "-NOT AVAILABLE-")].copy()
print("Unique cleaned organizations by category:")
print(df_work.drop_duplicates("clean")["entity_type"].value_counts())

Unique cleaned organizations by category:
entity_type
company             1658
university           741
gov / non-profit     250
uncertain              1
Name: count, dtype: int64


## 2. Corpus Statistics & Semantic Embeddings
Compute squared inverse document frequency weights ($\text{idf}(t)^2$) across the sector corpus and encode unique cleaned organization names once (or reuse cached embeddings if already in memory).

In [4]:
BOILERPLATE_CORE = {
    "UNIV", "INST", "COLLEGE", "ACADEMY", "SCHOOL", "HOCHSCHULE", "ECOLE", "POLITECNICA", "POLYTECHNICAL",
    "TECH", "SCIENCE", "RESEARCH", "LAB", "CENTER", "GROUP", "CORP", "COMPANY", "HOLDINGS", "INTERNATIONAL",
    "NATIONAL", "STATE", "PUBLIC", "ROYAL", "NORMAL", "GENERAL", "CENTRAL", "FEDERAL",
    "KORPERSCHAFT", "OFFENTLICHEN", "RECHTS", "BOARD", "REGENTS", "TRUSTEES", "FOUNDATION", "ASSOCIATION",
    "QUANTUM", "COMPUTING", "COMPUTER", "INFORMATION", "SYSTEMS", "NETWORK", "ELECTRONIC", "ELECTRONICS",
    "ELECTRIC", "POWER", "GRID", "ENERGY", "DIGITAL", "INTELLIGENT", "SMART", "DATA", "SOFTWARE",
    "CHINA", "CHINESE", "BEIJING", "SHANGHAI", "SHENZHEN", "GUANGDONG", "ANHUI", "JIANGSU", "ZHEJIANG",
    "HEFEI", "NANJING", "SUZHOU", "HANGZHOU", "WUHAN", "CHENGDU", "CHONGQING", "DALIAN", "HARBIN",
    "NORTH", "SOUTH", "EAST", "WEST", "NORTHEAST", "NORTHWEST", "SOUTHEAST", "SOUTHWEST", "NEW",
    "QUANTIQUE", "TIME", "DEGLI", "STUDI", "NORMALE", "SUPERIEURE", "POSTS",
    "COOPERATION", "ACADEMIC", "ACADEMIA", "CIRCUITS", "INTEGRATED",
    "PLATFORMS", "PLATFORM", "MOTOR", "INTELLIGENCE", "ARTIFICIAL"
}

unique_clean_names = sorted(df_work["clean"].dropna().unique())
N_CORPUS = len(unique_clean_names)
df_counts = Counter(t for name in unique_clean_names for t in set(name.split()))
idf_sq = {t: (np.log(N_CORPUS / c)) ** 2 for t, c in df_counts.items()}

# Encode unique names (or reuse `names` and `emb` if already present in notebook memory)
if "names" in globals() and "emb" in globals() and set(unique_clean_names).issubset(set(names)):
    name_to_idx = {n: idx for idx, n in enumerate(names)}
    emb_matrix = emb[[name_to_idx[n] for n in unique_clean_names]]
    print(f"Reused cached embeddings for {N_CORPUS:,} unique names.")
else:
    sbert_model = SentenceTransformer(MODEL_NAME)
    emb_matrix = sbert_model.encode(unique_clean_names, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
    print(f"Encoded {N_CORPUS:,} unique names.")

clean_to_emb_idx = {n: idx for idx, n in enumerate(unique_clean_names)}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Encoded 2,650 unique names.


## 3. Rectification Scoring & Louvain Sub-Clustering
Define `refine_subcluster()`, which decomposes oversized `(harm_id, entity_type)` blocks using squared-IDF Jaccard, distinguishing-core guards, and tightened Louvain community detection.

In [5]:
def refine_subcluster(member_names, th=SCORE_THRESHOLD, res=LOUVAIN_RESOLUTION):
    m = len(member_names)
    if m < MIN_CLUSTER_SIZE_TO_SPLIT:
        return [set(member_names)]

    idxs = [clean_to_emb_idx[n] for n in member_names]
    cos_mat = emb_matrix[idxs] @ emb_matrix[idxs].T

    toks = [set(n.split()) for n in member_names]
    core_toks = [({t for t in tk if t not in BOILERPLATE_CORE} or tk) for tk in toks]
    core_strs = [" ".join(t for t in n.split() if t not in BOILERPLATE_CORE) or n for n in member_names]
    tok_sums = [sum(idf_sq.get(t, 0.0) for t in tk) for tk in toks]
    core_sums = [sum(idf_sq.get(t, 0.0) for t in tk) for tk in core_toks]

    G_sub = nx.Graph()
    G_sub.add_nodes_from(member_names)

    for i in range(m):
        for j in range(i + 1, m):
            cos = float(cos_mat[i, j])

            # Squared-IDF Jaccard on full cleaned tokens
            inter = toks[i] & toks[j]
            if inter:
                inter_w = sum(idf_sq.get(t, 0.0) for t in inter)
                jac = inter_w / max(tok_sums[i] + tok_sums[j] - inter_w, 1e-9)
            else:
                jac = 0.0

            # Distinguishing-core guard (prevents institutional/sector template merges)
            c_inter = core_toks[i] & core_toks[j]
            if c_inter:
                c_inter_w = sum(idf_sq.get(t, 0.0) for t in c_inter)
                c_jac = c_inter_w / max(core_sums[i] + core_sums[j] - c_inter_w, 1e-9)
            else:
                c_jac = 0.0
            c_ed = rf_token_sort(core_strs[i], core_strs[j]) / 100.0

            if c_jac < CORE_JAC_MIN and c_ed < CORE_EDIT_MIN:
                continue

            ed = rf_token_sort(member_names[i], member_names[j]) / 100.0
            score = (cos + jac + ed) / 3.0
            if score >= th:
                G_sub.add_edge(member_names[i], member_names[j], weight=score)

    return list(nx_comm.louvain_communities(G_sub, weight="weight", resolution=res, seed=LOUVAIN_SEED))

## 4. Execute Stage 2 Partitioning & Canonical Naming
Split every Stage 1 cluster by `entity_type`, refine oversized sub-clusters, and assign deterministic `HARM_XXXXX` IDs and frequency-weighted canonical names.

In [6]:
final_communities = []
for (hid, etype), grp in df_work.groupby(["harm_id", "entity_type"]):
    members = sorted(grp["clean"].dropna().unique())
    for sub_comm in refine_subcluster(members):
        final_communities.append((etype, sub_comm))

# Weight canonical name selection by frequency in the full raw PATSTAT dataset
raw_freq = raw_df.merge(df_work[["psn_name", "clean"]].drop_duplicates(), on="psn_name", how="inner")
clean_weights = raw_freq["clean"].value_counts().to_dict()

final_communities.sort(key=lambda item: (-sum(clean_weights.get(n, 0) for n in item[1]), min(item[1])))

clean_to_id, clean_to_canon, clean_to_type = {}, {}, {}
for idx, (etype, members) in enumerate(final_communities, start=1):
    new_id = f"HARM_{idx:05d}"
    best_name = min(members, key=lambda n: (-clean_weights.get(n, 0), len(n), n))
    for n in members:
        clean_to_id[n] = new_id
        clean_to_canon[n] = best_name
        clean_to_type[n] = etype

df_work["harm_id_final"] = df_work["clean"].map(clean_to_id)
df_work["harmonized_name_final"] = df_work["clean"].map(clean_to_canon)

print(f"Stage 1 clusters: {df_work['harm_id'].nunique():,} -> Stage 2 rectified clusters: {len(final_communities):,}")

Stage 1 clusters: 1,359 -> Stage 2 rectified clusters: 2,176


## 4B. Core-Anchor & Subsidiary Consolidation (Recall Recovery)

While Stage 2 decomposes oversized hairball clusters and prevents cross-sector collisions, strict lexical and Louvain resolution guards can split legitimate regional subsidiaries, word-order permutations, and acronym expansions of the same organization (e.g., `IBM` vs. `IBM DEUTSCHLAND` / `INTERNATIONAL BUSINESS MACHINES`, `TOSHIBA` vs. `TOSHIBA RESEARCH EUROPE`, `STANFORD UNIV` vs. `LELAND STANFORD JUNIOR UNIV`).

To recover recall without lowering the pairwise similarity threshold across the whole graph, this step consolidates Stage 2 clusters **within compatible entity categories** whenever their canonical names satisfy one of three high-precision rules:
1. **Rare Brand-Anchor Subset:** After stripping generic institutional, legal, sector, and geographic descriptors, one cluster's distinguishing core tokens are a strict subset of another's and share at least one rare proper identifier ($\text{df} \le 12$), while preventing universities from merging across distinct institutional types (`UNIV` vs. `POLYTECHNIC` / `SCHOOL MINES`).
2. **Core Spelling / Concatenation Match:** Distinguishing cores ($\ge 8$ characters) match with $\ge 90\%$ character similarity (`PSIQUANTUM` $\leftrightarrow$ `PSYQUANTUM`).
3. **Exact Acronym Expansion:** A 3–6 letter uppercase acronym matches the exact word initials of another entity in the same category (`IBM` $\leftrightarrow$ `INTERNATIONAL BUSINESS MACHINES`, `NTT` $\leftrightarrow$ `NIPPON TELEGRAPH TELEPHONE`).

In [7]:
US_STATES = {"COLORADO", "FLORIDA", "ARIZONA", "MICHIGAN", "OHIO", "PENN", "TEXAS", "CALIFORNIA", "WASHINGTON", "VIRGINIA"}

ANCHOR_STOPWORDS = {
    # Institutional & Legal
    "UNIV", "INST", "INSTITUTE", "COLLEGE", "ACADEMY", "SCHOOL", "HOCHSCHULE", "ECOLE",
    "POLITECNICA", "POLYTECHNICAL", "POLYTECHNIC", "UNIVERSITETT", "UNIVERSITAET", "UNIVERSITAT",
    "UNIVERSITE", "UNIVERSIDAD", "UNIVERSITA", "UNIVERSITET", "TECHNISCHE", "VOCATIONAL",
    "TECH", "SCIENCE", "RESEARCH", "LAB", "CENTER", "GROUP", "CORP", "COMPANY", "COLTD",
    "HOLDINGS", "INTERNATIONAL", "NATIONAL", "STATE", "PUBLIC", "ROYAL", "NORMAL", "GENERAL",
    "CENTRAL", "FEDERAL", "LIMITED", "RESPONSIBILITY", "KORPERSCHAFT", "OFFENTLICHEN", "RECHTS",
    "BOARD", "REGENTS", "TRUSTEES", "FOUNDATION", "ASSOCIATION", "PRESIDENT", "FELLOWS",
    "HOSPITAL", "DEPARTMENT", "MINISTRY", "GOVERNMENT", "AGENCY", "SECRETARY", "DIRECTOR",
    "COMMITTEE", "COUNCIL", "INSTITUTION", "ADMINISTRATIVE", "INDEPENDENT",
    "FEDERALNOE", "GOSUDARSTVENNOE", "BYUDZHETNOE", "OBRAZOVATELNOE", "UCHREZHDENIE", "VYSSHEGO", "OBRAZOVANIYA",
    "GESELLSCHAFT", "FORDERUNG", "ANGEWANDTEN", "FORSCHUNG", "WISSENSCHAFTEN", "EV",
    "DEGLI", "STUDI", "NORMALE", "SUPERIEURE", "POSTS", "COOPERATION", "ACADEMIC", "ACADEMIA",
    # Sector & Business Descriptors
    "QUANTUM", "QUANTIQUE", "COMPUTING", "COMPUTER", "CALCULATING", "INFORMATION", "SYSTEMS", "SYSTEM",
    "NETWORK", "NETWORKS", "ELECTRONIC", "ELECTRONICS", "ELECTRIC", "POWER", "GRID", "ENERGY",
    "DIGITAL", "INTELLIGENT", "SMART", "DATA", "SOFTWARE", "SOLUTIONS", "SOLUTION",
    "INFRASTRUCTURE", "INSTRUMENT", "INSTRUMENTS", "DEVICES", "LICENSING", "PERMISSION",
    "INNOVATION", "INNOVATIONS", "VENTURES", "CAPITAL", "INVESTMENT", "PORTFOLIO",
    "INTELLECTUAL", "PROPERTY", "FINANCIAL", "INDUSTRY", "INDUSTRIAL", "DEVELOPMENT",
    "TECHNOLOGY", "MANUFACTURING", "APPLICATION", "APPLIED", "SERVICES", "SERVICE",
    "MACHINES", "EQUIPMENT", "AEROSPACE", "SPACE", "VISION", "BIOLOGICAL", "HEALTH",
    "MEDICINE", "MEDICAL", "ATOMIC", "ALTERNATIVE", "MOBILE", "MOBILITY", "TELECOM",
    "TELECOMMUNICATIONS", "COMMUNICATION", "COMMUNICATIONS", "ENGINEERING", "AGRICULTURAL",
    "MARITIME", "PETROLEUM", "MINERALS", "JIAOTONG", "EDUCATION", "HIGHER", "OPTICS",
    "OPTO", "PHOTONICS", "NANO", "NANOSCIENCE", "NANOTECHNOLOGY", "TOOLS", "CLOUD",
    "STORAGE", "SECURITY", "DEFENCE", "DEFENSE", "MILITARY", "PLA", "ARMY", "NAVY", "FORCE",
    "MOTOR", "PLATFORMS", "PLATFORM", "INTELLIGENCE", "ARTIFICIAL", "CIRCUITS", "INTEGRATED", "TIME",
    "SILICON", "WAVE", "GLOBAL", "STAR", "GREAT", "SUPER", "ADVANCED", "FUTURE", "OPEN", "SECOND", "MATRIX",
    # Countries, Regions & Major Cities
    "CHINA", "CHINESE", "JAPAN", "KOREA", "EUROPE", "EUROPEAN", "AMERICA", "AMERICAN", "UNITED", "STATES",
    "KINGDOM", "UK", "US", "DEUTSCHLAND", "GERMANY", "FRANCE", "AUSTRALIA", "CANADA", "ISRAEL", "SWITZERLAND", "NETHERLANDS",
    "BEIJING", "SHANGHAI", "SHENZHEN", "GUANGDONG", "ANHUI", "JIANGSU", "ZHEJIANG", "SICHUAN", "HUBEI",
    "HUNAN", "HENAN", "HEBEI", "FUJIAN", "SHANDONG", "SHANXI", "SHAANXI", "LIAONING", "JILIN", "HEILONGJIANG",
    "YUNNAN", "GUIZHOU", "GANSU", "QINGHAI", "HAINAN", "NINGXIA", "XINJIANG", "XIZANG", "GUANGXI", "INNER", "MONGOLIA",
    "HEFEI", "NANJING", "SUZHOU", "HANGZHOU", "WUHAN", "CHENGDU", "CHONGQING", "DALIAN", "HARBIN",
    "TIANJIN", "XIAN", "CHANGCHUN", "SHENYANG", "ZHENGZHOU", "CHANGSHA", "FUZHOU", "XIAMEN", "JINAN",
    "KUNMING", "TAIYUAN", "DONGGUAN", "QINGDAO", "WUXI",
    "NORTH", "SOUTH", "EAST", "WEST", "NORTHEAST", "NORTHWEST", "SOUTHEAST", "SOUTHWEST", "NEW", "YORK",
    "OXFORD", "CAMBRIDGE", "TOKYO", "SEOUL", "PARIS", "BERLIN", "MUNICH", "MUNCHEN", "ZURICH", "VIENNA", "WIEN", "DELFT"
} | US_STATES

INST_TYPE_TOKENS = {"UNIV", "POLYTECHNIC", "COLLEGE", "SCHOOL", "MINES", "INST", "ACADEMY", "HOSPITAL"}

def _extract_anchor(name, etype):
    stops = (ANCHOR_STOPWORDS - US_STATES) if etype == "university" else ANCHOR_STOPWORDS
    return {t for t in str(name).split() if t not in stops and len(t) >= 3}

def _is_exact_acronym(acro, full_str):
    if not (3 <= len(acro) <= 6 and acro.isalpha() and len(acro.split()) == 1):
        return False
    words = [w for w in full_str.split() if w != acro]
    return len(words) == len(acro) and acro == "".join(w[0] for w in words)

stage2_clusters = (
    df_work.groupby(["harm_id_final", "entity_type", "harmonized_name_final"])
    .agg(weight=("clean", lambda s: sum(clean_weights.get(x, 1) for x in s)))
    .reset_index()
)

RARE_ANCHOR_IDF = np.log(N_CORPUS / 12.0)
G_anchor = nx.Graph()
G_anchor.add_nodes_from(stage2_clusters["harm_id_final"])

for etype, grp in stage2_clusters.groupby("entity_type"):
    if etype == "uncertain":
        continue
    rows = list(grp.itertuples())
    for i in range(len(rows)):
        a = rows[i].harmonized_name_final
        ca = _extract_anchor(a, etype)
        ca_str = " ".join(sorted(ca))
        ia = set(a.split()) & INST_TYPE_TOKENS
        for j in range(i + 1, len(rows)):
            b = rows[j].harmonized_name_final
            cb = _extract_anchor(b, etype)
            cb_str = " ".join(sorted(cb))
            ib = set(b.split()) & INST_TYPE_TOKENS

            if etype == "university" and ia and ib and ia != ib:
                continue

            do_merge = False
            if ca and cb and (ca <= cb or cb <= ca):
                shared = ca & cb
                if any(np.log(N_CORPUS / df_counts[t]) >= RARE_ANCHOR_IDF for t in shared):
                    do_merge = True

            if not do_merge and len(ca_str) >= 8 and len(cb_str) >= 8:
                if rf_token_sort(ca_str, cb_str) >= 90.0:
                    do_merge = True

            if not do_merge and (_is_exact_acronym(a, b) or _is_exact_acronym(b, a)):
                do_merge = True

            if do_merge:
                G_anchor.add_edge(rows[i].harm_id_final, rows[j].harm_id_final)

merged_components = list(nx.connected_components(G_anchor))
old_id_to_weight = stage2_clusters.set_index("harm_id_final")["weight"].to_dict()
old_id_to_canon = stage2_clusters.set_index("harm_id_final")["harmonized_name_final"].to_dict()

merged_components.sort(
    key=lambda comp: (-sum(old_id_to_weight[hid] for hid in comp), min(old_id_to_canon[hid] for hid in comp))
)

old_to_new_id = {}
for idx, comp in enumerate(merged_components, start=1):
    new_hid = f"HARM_{idx:05d}"
    for hid in comp:
        old_to_new_id[hid] = new_hid

df_work["harm_id_final"] = df_work["harm_id_final"].map(old_to_new_id)

def _pick_canonical(sub_clean):
    unique_members = sub_clean.unique()
    return min(unique_members, key=lambda s: (-clean_weights.get(s, 1), len(s), s))

canon_map = df_work.groupby("harm_id_final")["clean"].apply(_pick_canonical)
df_work["harmonized_name_final"] = df_work["harm_id_final"].map(canon_map)

print(f"Stage 3 Anchor Consolidation complete: {len(stage2_clusters):,} -> {df_work['harm_id_final'].nunique():,} final entities.")

Stage 3 Anchor Consolidation complete: 2,180 -> 1,985 final entities.


## 5. Quality Diagnostics & Export
Inspect the largest post-rectification clusters and save both the lookup dictionary (`harmonization_lookup_rectified.csv`) and the full enriched PATSTAT applicant table (`qc_applicants_harmonized_PROD.csv`).

In [8]:
cluster_audit = (
    df_work.groupby(["harm_id_final", "entity_type", "harmonized_name_final"])
    .agg(n_raw=("psn_name", "nunique"), n_clean=("clean", "nunique"))
    .reset_index()
    .sort_values("n_clean", ascending=False)
)

print("Cluster size distribution (n_clean):")
print(cluster_audit["n_clean"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("\nTop 20 largest harmonized organizations:")
print(cluster_audit.head(20).to_string(index=False))

# Export lookup table
lookup_out = (
    df_work[["psn_name", "clean", "entity_type", "harm_id_final", "harmonized_name_final"]]
    .drop_duplicates()
    .sort_values(["harm_id_final", "psn_name"])
)
lookup_out.to_csv(OUTPUT_LOOKUP_CSV, index=False, encoding="utf-8-sig")

# Merge back onto full PATSTAT applicant table
full_out = raw_df.merge(
    lookup_out[["psn_name", "clean", "entity_type", "harm_id_final", "harmonized_name_final"]].drop_duplicates("psn_name"),
    on="psn_name",
    how="inner"
)
full_out.to_csv(OUTPUT_FULL_CSV, index=False, encoding="utf-8-sig")
print(f"\nSaved lookup ({len(lookup_out):,} rows) -> {OUTPUT_LOOKUP_CSV}")
print(f"Saved full dataset ({len(full_out):,} rows) -> {OUTPUT_FULL_CSV}")

Cluster size distribution (n_clean):
count    1993.000000
mean        1.333668
std         1.020648
min         1.000000
50%         1.000000
90%         2.000000
95%         3.000000
99%         5.000000
max        18.000000
Name: n_clean, dtype: float64

Top 20 largest harmonized organizations:
harm_id_final      entity_type                                               harmonized_name_final  n_raw  n_clean
   HARM_00004          company                                 IBM INTERNATIONAL BUSINESS MACHINES     20       18
   HARM_00051          company                                      SUZHOU INSPUR INTELLIGENT TECH     12       11
   HARM_00237          company INFORMATION COMMUNICATION BRANCH STATE GRID ZHEJIANG ELECTRIC POWER     11       11
   HARM_00001          company                                 ORIGIN QUANTUM COMPUTING TECH HEFEI     21       11
   HARM_00028          company                                          ALIBABA DAMO HANGZHOU TECH     14       10
   HARM_0002